# Notebook 03 - Transformations métier et contrôles finaux

## Objectif

Le staging a rendu la donnée lisible.  
La transformation va maintenant la rendre **utile pour les KPI**.

## Différence entre staging et transformation

- **Staging** : rendre la donnée techniquement exploitable
- **Transformation métier** : choisir ce qui sert le besoin analytique

## Ce qu'on va produire

- `tr_notice_kpi_base`
- `tr_notice_cpv`

## Pourquoi cette étape existe

Un junior fait souvent l'erreur suivante :

> "J'ai déjà une table standardisée, donc je peux charger directement le DWH."

En réalité, il manque encore :

- la déduplication métier,
- l'idempotence,
- les colonnes dérivées utiles aux KPI,
- la réduction du périmètre aux colonnes vraiment nécessaires.

In [1]:
# Nous importons pandas.
import pandas as pd

# Nous importons la bibliothèque pathlib pour manipuler les chemins de fichiers proprement.
from pathlib import Path

In [2]:
# Nous définissons un chemin de départ qui correspond au dossier courant du notebook.
current_dir = Path.cwd().resolve()

# Si le notebook est exécuté depuis le dossier du projet, ce test sera vrai.
if (current_dir / "data").exists():
    # Dans ce cas, le dossier courant est déjà la racine du projet.
    project_root = current_dir
else:
    # Sinon, on suppose que le notebook est exécuté depuis le dossier notebooks.
    project_root = current_dir.parent

# Nous définissons ici le dossier qui contient les lots bruts.
data_dir = project_root / "data" / "raw_batches"

# Nous définissons ici le dossier de sortie.
output_dir = project_root / "outputs"

# Nous affichons la racine retenue pour éviter toute ambiguïté.
print("project_root =", project_root)

# Nous affichons la liste des fichiers bruts disponibles.
print("raw files =", sorted(p.name for p in data_dir.glob("*.json")))

project_root = D:\Ens-EPSI\boamp_etl_dwh_tiny_explained_project
raw files = ['B20150308_01_initial.json', 'B20150309_01_incremental.json', 'B20150309_01_rerun.json']


## Étape 1 - Charger les sorties de staging

In [3]:
# Nous lisons la table standardisée des avis.
stg_notice_std = pd.read_csv(output_dir / "staging" / "stg_notice_std.csv")

# Nous lisons la table CPV du staging.
stg_notice_cpv = pd.read_csv(output_dir / "staging" / "stg_notice_cpv.csv")

# Nous convertissons les timestamps pour garder les bons types.
stg_notice_std["publication_ts"] = pd.to_datetime(stg_notice_std["publication_ts"], utc=True)
stg_notice_std["deadline_ts"] = pd.to_datetime(stg_notice_std["deadline_ts"], utc=True)

# Nous affichons les tables chargées.
display(stg_notice_std)
display(stg_notice_cpv)

,batch_id,record_hash,idweb,objet,buyer_name,publication_ts,deadline_ts,response_delay_days,market_type_code,url_avis,has_deadline_flag
0,B20150308_01_initial,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,15-31501,concession d'aménagement pour la réalisation d...,Cté Cnes Moyenne Vilaine et Semnon,2015-03-03 00:00:00+00:00,2015-04-20 11:00:00+00:00,48,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,True
1,B20150308_01_initial,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,15-40000,maintenance de bâtiments scolaires et travaux ...,Ville de Rennes,2015-03-04 00:00:00+00:00,2015-03-18 12:00:00+00:00,14,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,True
2,B20150308_01_initial,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,15-40001,fourniture de matériel informatique pour les s...,Ville de Nantes,2015-03-05 00:00:00+00:00,2015-03-25 16:00:00+00:00,20,FOURNITURES,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,True
3,B20150308_01_initial,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,15-40002,prestations de nettoyage des locaux administra...,Conseil départemental de Loire-Atlantique,2015-03-06 00:00:00+00:00,2015-03-20 10:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,True
4,B20150309_01_incremental,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,Métropole de Lyon,2015-03-07 00:00:00+00:00,2015-03-30 15:00:00+00:00,23,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,True
5,B20150309_01_rerun,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,Métropole de Lyon,2015-03-07 00:00:00+00:00,2015-03-30 15:00:00+00:00,23,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,True
6,B20150309_01_incremental,1183366902af82c4fce3ae680bbe556987e3d1100b9f5c...,15-40004,services d'assistance juridique en urbanisme,Communauté urbaine de Bordeaux,2015-03-08 00:00:00+00:00,2015-03-22 09:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,True
7,B20150309_01_rerun,1183366902af82c4fce3ae680bbe556987e3d1100b9f5c...,15-40004,services d'assistance juridique en urbanisme,Communauté urbaine de Bordeaux,2015-03-08 00:00:00+00:00,2015-03-22 09:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,True


,idweb,cpv_code,is_primary_flag
0,15-31501,45111291,True
1,15-31501,71000000,False
2,15-31501,70122100,False
3,15-40000,45453100,True
4,15-40000,50700000,False
5,15-40001,30200000,True
6,15-40001,48000000,False
7,15-40002,90910000,True
8,15-40003,45233140,True
9,15-40003,34992200,False


## Étape 2 - Déduplication et idempotence

Nous avons volontairement un batch `RERUN`.

L'objectif est de montrer qu'une ré-exécution ne doit pas créer de doublons métier.

### Règle choisie

On garde une seule ligne métier par `idweb`, en appliquant deux protections :

1. on enlève les doublons exacts `idweb + record_hash`
2. on garde ensuite la dernière version métier par `idweb`

In [4]:
# Nous trions d'abord les lignes pour avoir une règle déterministe.
tr_notice = stg_notice_std.sort_values(["publication_ts", "batch_id", "idweb"])

# Nous retirons les doublons stricts de payload sur la clé `idweb + record_hash`.
tr_notice = tr_notice.drop_duplicates(subset=["idweb", "record_hash"])

# Nous retirons ensuite les doublons métier éventuels sur `idweb`.
tr_notice = tr_notice.drop_duplicates(subset=["idweb"], keep="last")

# Nous réindexons proprement la table.
tr_notice = tr_notice.reset_index(drop=True)

# Nous affichons le résultat.
tr_notice

,batch_id,record_hash,idweb,objet,buyer_name,publication_ts,deadline_ts,response_delay_days,market_type_code,url_avis,has_deadline_flag
0,B20150308_01_initial,510aba836dde6afdee6d4a1d83a3747b5c89ebd2e6c4f0...,15-31501,concession d'aménagement pour la réalisation d...,Cté Cnes Moyenne Vilaine et Semnon,2015-03-03 00:00:00+00:00,2015-04-20 11:00:00+00:00,48,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,True
1,B20150308_01_initial,283b0d32917fa437e5761988a87d21242e8c0fefca0203...,15-40000,maintenance de bâtiments scolaires et travaux ...,Ville de Rennes,2015-03-04 00:00:00+00:00,2015-03-18 12:00:00+00:00,14,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,True
2,B20150308_01_initial,13848b9951b0faafb1c5dc92917c9e15982abbc6599191...,15-40001,fourniture de matériel informatique pour les s...,Ville de Nantes,2015-03-05 00:00:00+00:00,2015-03-25 16:00:00+00:00,20,FOURNITURES,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,True
3,B20150308_01_initial,66674b9826d0cf1f35b23687d7ec4d9fab1cd75c0c1bdb...,15-40002,prestations de nettoyage des locaux administra...,Conseil départemental de Loire-Atlantique,2015-03-06 00:00:00+00:00,2015-03-20 10:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,True
4,B20150309_01_incremental,3bb0e6ed1ffaad43668165435dd93e592aed0402179744...,15-40003,travaux d'aménagement de voirie et signalisation,Métropole de Lyon,2015-03-07 00:00:00+00:00,2015-03-30 15:00:00+00:00,23,TRAVAUX,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,True
5,B20150309_01_incremental,1183366902af82c4fce3ae680bbe556987e3d1100b9f5c...,15-40004,services d'assistance juridique en urbanisme,Communauté urbaine de Bordeaux,2015-03-08 00:00:00+00:00,2015-03-22 09:00:00+00:00,14,SERVICES,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,True


## Étape 3 - Construire la base analytique minimale

Nous n'allons pas envoyer toute la table dans le DWH.  
Nous n'en gardons que le minimum utile pour les KPI.

### Colonnes retenues

- `idweb`
- `batch_id`
- `publication_date`
- `buyer_name`
- `market_type_code`
- `response_delay_days`
- `has_deadline_flag`
- `url_avis`
- `objet_court`
- `notice_count`

### Pourquoi ce choix est réaliste

On garde les colonnes qui servent :

- le grain,
- les jointures,
- les KPI,
- l'audit minimal.

In [5]:
# Nous dérivons la date civile de publication.
tr_notice["publication_date"] = pd.to_datetime(tr_notice["publication_ts"]).dt.date

# Nous dérivons un objet court pour les affichages éventuels.
tr_notice["objet_court"] = tr_notice["objet"].fillna("").str.slice(0, 100)

# Nous ajoutons la mesure la plus simple du monde décisionnel : 1 ligne = 1 avis.
tr_notice["notice_count"] = 1

# Nous sélectionnons uniquement les colonnes utiles aux KPI.
tr_notice_kpi_base = tr_notice[
    [
        "idweb",
        "batch_id",
        "publication_date",
        "buyer_name",
        "market_type_code",
        "response_delay_days",
        "has_deadline_flag",
        "url_avis",
        "objet_court",
        "notice_count",
    ]
].reset_index(drop=True)

# Nous affichons le résultat.
tr_notice_kpi_base

,idweb,batch_id,publication_date,buyer_name,market_type_code,response_delay_days,has_deadline_flag,url_avis,objet_court,notice_count
0,15-31501,B20150308_01_initial,2015-03-03,Cté Cnes Moyenne Vilaine et Semnon,TRAVAUX,48,True,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,concession d'aménagement pour la réalisation d...,1
1,15-40000,B20150308_01_initial,2015-03-04,Ville de Rennes,TRAVAUX,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,maintenance de bâtiments scolaires et travaux ...,1
2,15-40001,B20150308_01_initial,2015-03-05,Ville de Nantes,FOURNITURES,20,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,fourniture de matériel informatique pour les s...,1
3,15-40002,B20150308_01_initial,2015-03-06,Conseil départemental de Loire-Atlantique,SERVICES,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,prestations de nettoyage des locaux administra...,1
4,15-40003,B20150309_01_incremental,2015-03-07,Métropole de Lyon,TRAVAUX,23,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,travaux d'aménagement de voirie et signalisation,1
5,15-40004,B20150309_01_incremental,2015-03-08,Communauté urbaine de Bordeaux,SERVICES,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,services d'assistance juridique en urbanisme,1


## Étape 4 - Filtrer la table CPV sur les avis réellement retenus

Pourquoi cette étape est importante ?

Si une ligne notice a été rejetée ou supprimée par déduplication, ses CPV ne doivent pas survivre seuls.

In [6]:
# Nous filtrons les CPV pour ne garder que ceux des avis présents dans la base KPI.
tr_notice_cpv = stg_notice_cpv[stg_notice_cpv["idweb"].isin(tr_notice_kpi_base["idweb"])]

# Nous supprimons les doublons par sécurité.
tr_notice_cpv = tr_notice_cpv.drop_duplicates().reset_index(drop=True)

# Nous affichons le résultat.
tr_notice_cpv

,idweb,cpv_code,is_primary_flag
0,15-31501,45111291,True
1,15-31501,71000000,False
2,15-31501,70122100,False
3,15-40000,45453100,True
4,15-40000,50700000,False
5,15-40001,30200000,True
6,15-40001,48000000,False
7,15-40002,90910000,True
8,15-40003,45233140,True
9,15-40003,34992200,False


## Étape 5 - Sauvegarder les sorties de transformation

In [7]:
# Nous définissons le dossier des transformations.
transform_dir = output_dir / "transformations"

# Nous créons le dossier si besoin.
transform_dir.mkdir(parents=True, exist_ok=True)

# Nous écrivons la base analytique des avis.
tr_notice_kpi_base.to_csv(transform_dir / "tr_notice_kpi_base.csv", index=False)

# Nous écrivons la table transformée des CPV.
tr_notice_cpv.to_csv(transform_dir / "tr_notice_cpv.csv", index=False)

# Nous affichons les fichiers produits.
sorted(p.name for p in transform_dir.glob("*.csv"))

['tr_notice_cpv.csv', 'tr_notice_kpi_base.csv']

## Auto-critique de design

### Pourquoi on ne charge pas directement les textes longs dans la fact

Parce qu'une fact doit rester compacte, stable et mesurable.

### Pourquoi on ne garde pas toutes les colonnes BOAMP

Parce que nos KPI n'en ont pas besoin.

### Pourquoi on garde `notice_count = 1`

Parce que c'est la mesure la plus robuste pour compter les avis sans ambiguïté.

### Pourquoi `response_delay_days` est accepté dans la fact

Parce que c'est une mesure simple, numérique, agrégable en moyenne.

### Pourquoi les CPV restent séparés

Parce qu'un avis peut avoir plusieurs CPV.  
Les mettre dans la fact principale casserait le grain.
